# Global Crypto Market Dynamics & Liquidity 2026
## 1,000-Asset Empirical Telemetry, Turnover Regimes & Catastrophic Tail-Risk Forecasting

> **ElmasCore Financial Telemetry Monograph** • Grounded in CoinGecko Market Telemetry & Dual-Tier Cognitive CoT Annotation (Cohen's Kappa $\kappa = 0.9407$).

---

### Research Agenda
1. **Microstructural Census**: Exhaustive empirical mapping of the top 1,000 cryptocurrency assets ($2.84T aggregate market cap, 57.72% BTC dominance).
2. **Turnover Velocity & Slippage Modeling**: Formulating and classifying 4 structural order book liquidity regimes (`Hyper_Liquid_Turnover`, `High_Liquid_Active`, `Moderate_Liquid`, `Illiquid_Constrained`) via an ensemble of LightGBM and CatBoost.
3. **Catastrophic Tail Risk (>85% Peak Drawdown)**: Forecasting non-linear cyclical wipeouts and tokenomic dilution overhangs with calibrated Bayesian certainty.
4. **Explainability & Factor Attribution**: SHAP values and feature sensitivity across market depth, volume velocity, and unlock inflation.

In [1]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np
from IPython.display import Image, display

# Configure project root
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.core.config import ProjectConfig
from src.data_pipeline.loader import load_raw_telemetry
from src.data_pipeline.schema_validator import validate_telemetry_schema
from src.feature_engineering.microstructure_features import compute_microstructure_features
from src.feature_engineering.encoder import build_feature_matrix

cfg = ProjectConfig()
print(f"ElmasCore GCMDL Pipeline Loaded. Project Root: {PROJECT_ROOT}")

## 1. Census Ingestion & Empirical Summary

We ingest the top 1,000 cryptocurrency assets, ranked by market capitalization and verified against the official schema validator.

In [2]:
raw_df = load_raw_telemetry(cfg.paths.raw_csv)
is_valid, validation_errors = validate_telemetry_schema(raw_df)

print(f"Total Assets: {len(raw_df):,}")
print(f"Schema Validated: {is_valid} (Errors: {len(validation_errors)})")
print(f"Total Market Cap: ${raw_df['market_cap'].sum() / 1e12:.2f}T USD")
print(f"Total 24h Volume: ${raw_df['total_volume'].sum() / 1e9:.2f}B USD")
btc_dominance = (raw_df.loc[raw_df['symbol'] == 'BTC', 'market_cap'].iloc[0] / raw_df['market_cap'].sum()) * 100
print(f"Bitcoin (BTC) Market Dominance: {btc_dominance:.2f}%")

top_5 = raw_df[['symbol', 'name', 'current_price', 'market_cap', 'total_volume', 'market_cap_tier']].head(5)
display(top_5)

## 2. Microstructural Feature Engineering

We calculate mathematically grounded liquidity, volatility, and dilution indicators:
- **Turnover Ratio**: $V = \text{Volume}_{24h} / \text{MarketCap}$
- **Amihud Illiquidity**: $I_{\text{Amihud}} = \frac{|\Delta P / P|}{\text{Volume}_{24h}}$
- **Corwin-Schultz Volatility Spread**: High-Low volatility variance proxy
- **FDV Overhang Multiplier**: $\text{FDV} / \text{MarketCap}$
- **Circulating Supply Ratio**: $\text{Circulating Supply} / \text{Total Supply}$

In [3]:
engineered_df = compute_microstructure_features(raw_df)
matrix = build_feature_matrix(engineered_df)

print(f"Engineered Features Count: {matrix.features.shape[1]}")
print(f"Feature Names: {matrix.feature_names}")

# Display key features for top 5 assets
engineered_preview = engineered_df[[
    'symbol', 'turnover_ratio', 'amihud_illiquidity_log',
    'corwin_schultz_spread', 'circulating_supply_ratio', 'fdv_overhang_multiplier'
]].head(5)
display(engineered_preview)

## 3. Task 1: Multi-Class Liquidity Turnover Regime Classification

Classification into 4 regimes:
- `Illiquid_Constrained` (<0.02 turnover)
- `Moderate_Liquid` (0.02 - 0.10)
- `High_Liquid_Active` (0.10 - 0.30)
- `Hyper_Liquid_Turnover` (>0.30)

In [4]:
with open(cfg.paths.artifacts_dir / 'metrics_summary.json', 'r') as f:
    summary = json.load(f)

task1 = summary['turnover_classification']
print('=== Task 1: Turnover Regime Classification (5-Fold CV Ensemble) ===')
print(f"Macro F1:            {task1['macro_f1']:.4f}")
print(f"Weighted F1:         {task1['weighted_f1']:.4f}")
print(f"Balanced Accuracy:   {task1['balanced_accuracy'] * 100:.2f}%")
print(f"Multi-Class LogLoss: {task1['multi_class_log_loss']:.4f}")
print(f"Brier Multi-Score:   {task1['brier_score']:.4f}")
print('\nPer-Class F1 Scores:')
for cls_name, f1_val in task1['per_class_f1'].items():
    print(f"  - {cls_name:24s}: {f1_val:.4f}")

## 4. Task 2: Tail-Risk Catastrophic Drawdown Prediction

Predicting whether an asset experiences an extreme drawdown exceeding 85% from its All-Time High.

In [5]:
task2 = summary['tail_risk_prediction']
print('=== Task 2: Catastrophic Drawdown (>85%) Prediction ===')
print(f"ROC-AUC:           {task2['roc_auc']:.4f}")
print(f"PR-AUC:            {task2['pr_auc']:.4f}")
print(f"Optimal Threshold: {task2['optimal_threshold']:.4f}")
print(f"Sensitivity:       {task2['sensitivity'] * 100:.2f}%")
print(f"Specificity:       {task2['specificity'] * 100:.2f}%")
print(f"Brier Score:       {task2['brier_score']:.4f}")

## 5. Visual Artifacts & Empirical Telemetry Figures

Below are the publication-grade figures produced by the ElmasCore telemetry suite.

In [6]:
img_dir = cfg.paths.images_dir
print('1. Microstructure Map:')
display(Image(filename=str(img_dir / 'liquidity_microstructure_map.png')))
print('2. Confusion Matrix:')
display(Image(filename=str(img_dir / 'confusion_matrix_turnover.png')))
print('3. Catastrophic Drawdown ROC and PR Curves:')
display(Image(filename=str(img_dir / 'catastrophic_drawdown_roc_pr.png')))
print('4. Tokenomics Dilution Overhang:')
display(Image(filename=str(img_dir / 'tokenomics_dilution_overhang.png')))
print('5. SHAP Feature Importance Attribution:')
display(Image(filename=str(img_dir / 'feature_importance_shap.png')))

## 6. Single-Asset Triage CLI Demonstration

We can query any cryptocurrency asset in the 1,000-asset census using the standardized single-asset triage interface.

In [7]:
from predict import find_asset_by_symbol, compute_asset_triage

for sym in ['BTC', 'ETH', 'SOL']:
    res = compute_asset_triage(sym, engineered_df)
    if res:
        print(f"=== {res['name']} ({res['symbol']}) ===")
        print(f"  Rank: #{res['rank']} | Price: ${res['price']:,.2f} | Market Cap: ${res['market_cap']:,.0f}")
        print(f"  Turnover Velocity: {res['turnover_ratio']:.4f} -> {res['turnover_regime']}")
        print(f"  ATH Drawdown: {res['ath_drawdown_pct']:.2f}% | Catastrophic Risk: {res['catastrophic_risk_prob']*100:.1f}%")
        print(f"  Circulating Supply Ratio: {res['circulating_supply_ratio']*100:.1f}%")
        print()